# Duty cycle experiment B — does the training duty-cycle mix change generalization?

**Question:** if the U-Net only sees low-duty or only high-duty modulated RFI during training, how well does it detect RFI at the duty cycles it never saw?

**Design:**
- **Shared base:** the 60 starter training recordings that are *not* duty-controlled modulations (carrier, radar, impulse, mixed, wireless, digital, clean, sky).
- **Swapped part:** 40 generated modulated recordings (switched_carrier, bpsk, qpsk, ask, fsk). Every condition uses the **same** 40 scene seeds, families and peak strengths; only the duty cycle differs. Total is 100 recordings = 1,000 antenna pairs, as in the grid.

| condition | training duty cycles |
|---|---|
| low  | 0.1, 0.25 |
| high | 0.75, 1.0 |
| all  | 0.1, 0.25, 0.5, 0.75, 1.0 |

- Same recipe as the best grid trial: U-Net/8, BCE, lr 1e-3, batch 1, normalization from the clean training recordings, best epoch by validation pixel AP, stop after ≥20 epochs and 10 epochs without a 0.001 AP gain. Each condition is trained with two initialization seeds, because seed-to-seed variance can be as large as the effect.
- **Evaluation:** a generated duty sweep (0.1–1.0, including 0.5 and 0.9, which low/high never see) with scene seeds disjoint from training. Metric is threshold-free pixel AP, compared between conditions at the same duty/strength.

**Caveats:** high-duty training sets contain more RFI pixels, so "more duty" also means "more positive examples". Early stopping uses the starter validation set, which mixes all duty levels.

## 1. Settings

In [ ]:
import os
import sys
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from cs163_rfi import paths  # folders come from .env at the repo root
DATA = paths.DATA_DIR
OUT_DIR = Path(os.environ.get("RFI_DUTY_B_DIR", paths.RESULTS_DIR / "duty_cycle/B_train_mix"))
CHECKPOINT_DIR = paths.CHECKPOINT_DIR / "duty_cycle" / OUT_DIR.name  # best.pt per run, same folder names as OUT_DIR
CONDITIONS = {"low": (0.1, 0.25), "high": (0.75, 1.0), "all": (0.1, 0.25, 0.5, 0.75, 1.0)}
TRAIN_SEEDS = (163, 164)  # model initialization / shuffle seeds
SYNTH_SEEDS = tuple(range(700001, 700041))  # 40 generated training scenes; starter seeds are 96000-120039
STRENGTH_RANGE = (0.01, 2.0)  # log-uniform peak strength, the range of the starter modulated recordings
EVAL_FAMILIES = ("switched_carrier", "bpsk", "fsk")
EVAL_DUTIES = (0.1, 0.25, 0.5, 0.75, 0.9, 1.0)
EVAL_STRENGTHS = (0.1, 0.5, 2.0)
EVAL_SEEDS = tuple(range(900001, 900005))
BASE_CHANNELS = 8
LEARNING_RATE = 1e-3
BATCH_SIZE = 1
MIN_EPOCHS = 20
PATIENCE = 10
MIN_AP_IMPROVEMENT = 0.001
MAX_EPOCHS = 100
AP_BINS = 65536  # validation AP, as in the grid
SWEEP_BINS = 4096

In [ ]:
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from IPython.display import display
from cs163_rfi import load_recording, simulate_recording
from cs163_rfi.simulation import MODULATIONS

plt.rcParams.update({"font.size": 11, "figure.dpi": 110,
                     "axes.spines.top": False, "axes.spines.right": False})
OUT_DIR.mkdir(parents=True, exist_ok=True)
catalog = pd.read_csv(DATA / "dataset.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "· output:", OUT_DIR, "· checkpoints:", CHECKPOINT_DIR)
if device.type == "cpu": print("Training on CPU is very slow; submit slurm/run_duty_cycle_B.sh.")

## 2. Model and helpers

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.layers(x)

class SegmentationModel(nn.Module):
    """Same two-level U-Net as RFI_unet_team.ipynb."""
    def __init__(self, base=8, use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        self.enc1 = DoubleConv(1, base)
        self.enc2 = DoubleConv(base, 2*base)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(2*base, 4*base)
        self.up2 = nn.ConvTranspose2d(4*base, 2*base, 2, stride=2)
        self.dec2 = DoubleConv(4*base if use_skips else 2*base, 2*base)
        self.up1 = nn.ConvTranspose2d(2*base, base, 2, stride=2)
        self.dec1 = DoubleConv(2*base if use_skips else base, base)
        self.output = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        z = self.bottleneck(self.pool(e2))
        z = self.up2(z)
        z = self.dec2(torch.cat((z, e2), dim=1) if self.use_skips else z)
        z = self.up1(z)
        z = self.dec1(torch.cat((z, e1), dim=1) if self.use_skips else z)
        return self.output(z)

def histogram_ap(positive, negative):
    """Pixel AP from quantized-score histograms (counts are sample weights)."""
    if positive.sum() == 0: return np.nan
    bins = len(positive)
    labels = np.r_[np.ones(bins), np.zeros(bins)]
    scores = np.tile(np.arange(bins), 2)
    weights = np.r_[positive, negative]
    keep = weights > 0
    return float(average_precision_score(labels[keep], scores[keep], sample_weight=weights[keep]))

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

def to_pairs(power, mask):
    """(time, antenna, channel, 1) recording -> one uint8 image and Boolean mask per antenna."""
    return [(power[:, a, :, 0].astype(np.uint8), mask[:, a, :, 0]) for a in range(power.shape[1])]

def load_pairs(frame):
    pairs = []
    for row in frame.itertuples():
        rec = load_recording(DATA / row.path)
        pairs += to_pairs(rec["power"], rec["rfi_mask"])
    return pairs

class PairDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, index):
        power, mask = self.pairs[index]
        x = (power.astype(np.float32) - NORM["mean"]) / NORM["std"]
        return torch.from_numpy(x)[None], torch.from_numpy(mask.astype(np.float32))[None]

def synthetic_pairs(duties):
    """Same scenes, families and strengths for every condition; only the duty cycle changes."""
    pairs, rows = [], []
    for i, seed in enumerate(SYNTH_SEEDS):
        rng = np.random.default_rng(seed)
        strength = float(np.exp(rng.uniform(*np.log(STRENGTH_RANGE))))
        family = MODULATIONS[i % len(MODULATIONS)]
        duty = duties[(i // len(MODULATIONS)) % len(duties)]  # each family meets every duty level
        rec = simulate_recording(seed=seed, rfi=family, strength=strength, duty_cycle=duty)
        pairs += to_pairs(rec["power"], rec["rfi_mask"])
        rows.append(dict(seed=seed, family=family, strength=strength, duty_cycle=duty,
                         labelled_fraction=float(rec["rfi_mask"].mean())))
    return pairs, pd.DataFrame(rows)

## 3. Load the shared data
Normalization is the grid's: one mean/std pooled over the five clean training recordings.

In [ ]:
clean = catalog.query("split == 'train' and rfi_type == 'clean'")
values = np.concatenate([load_recording(DATA / r.path)["power"].ravel() for r in clean.itertuples()]).astype(np.float64)
NORM = {"mean": float(values.mean()), "std": float(values.std())}
np.testing.assert_allclose([NORM["mean"], NORM["std"]], [4.257218322753906, 4.2564571117143295], rtol=1e-8)
del values

base_frame = catalog.query("split == 'train' and rfi_type not in @MODULATIONS")
base_pairs = load_pairs(base_frame)
val_pairs = load_pairs(catalog.query("split == 'validation'"))
val_loader = DataLoader(PairDataset(val_pairs), batch_size=10, shuffle=False)
print(f"Base training recordings: {len(base_frame)} ({len(base_pairs)} pairs); validation pairs: {len(val_pairs)}")
display(base_frame.rfi_type.value_counts().rename("base recordings").to_frame().T)

# Preview what each condition adds (this is the positive-pixel confound).
previews = {name: synthetic_pairs(duties)[1].assign(condition=name) for name, duties in CONDITIONS.items()}
preview = pd.concat(previews.values())
preview.to_csv(OUT_DIR / "synthetic_training_sets.csv", index=False)
display(preview.groupby("condition").agg(recordings=("seed", "size"), mean_duty=("duty_cycle", "mean"),
                                         mean_labelled_fraction=("labelled_fraction", "mean")))
del previews

## 4. Train every condition × seed
A finished run writes `done.json` and is skipped if the job is resubmitted. An unfinished run restarts from epoch 1.

In [ ]:
@torch.inference_mode()
def validation_ap(model):
    model.eval()
    positive = np.zeros(AP_BINS, np.int64); negative = np.zeros(AP_BINS, np.int64)
    for x, y in val_loader:
        q = np.floor(model(x.to(device)).sigmoid().cpu().numpy() * (AP_BINS - 1)).astype(np.int64)
        t = y.numpy().astype(bool)
        positive += np.bincount(q[t], minlength=AP_BINS); negative += np.bincount(q[~t], minlength=AP_BINS)
    return histogram_ap(positive, negative)

def train_condition(name, duties, seed):
    run_dir = OUT_DIR / f"{name}_seed{seed}"
    if (run_dir / "done.json").is_file():
        print(f"{run_dir.name}: already trained; skipping.")
        return json.loads((run_dir / "done.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = CHECKPOINT_DIR / run_dir.name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    synth, synth_table = synthetic_pairs(duties)
    synth_table.to_csv(run_dir / "synthetic_train.csv", index=False)
    seed_everything(seed)
    model = SegmentationModel(BASE_CHANNELS).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    train_loader = DataLoader(PairDataset(base_pairs + synth), batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(seed))
    assert len(train_loader.dataset) == 1000
    history, best_ap, best_epoch, anchor_ap, anchor_epoch = [], -np.inf, 0, -np.inf, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        started = time.perf_counter()
        model.train(); loss_sum = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = nn.functional.binary_cross_entropy_with_logits(model(x), y)
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            loss_sum += loss.item()
        val_ap = validation_ap(model)
        history.append(dict(epoch=epoch, train_loss=loss_sum / len(train_loader), validation_ap=val_ap,
                            seconds=time.perf_counter() - started))
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            torch.save(model.state_dict(), ckpt_dir / "best.pt")
        if val_ap > anchor_ap + MIN_AP_IMPROVEMENT:
            anchor_ap, anchor_epoch = val_ap, epoch
        print(f"{run_dir.name} epoch {epoch}: loss={history[-1]['train_loss']:.5f}, val AP={val_ap:.4f}", flush=True)
        if epoch >= MIN_EPOCHS and epoch - anchor_epoch >= PATIENCE: break
    status = dict(condition=name, train_seed=seed, duties=list(duties), best_epoch=best_epoch,
                  best_validation_ap=best_ap, last_epoch=epoch, converged=epoch - anchor_epoch >= PATIENCE)
    (run_dir / "done.json").write_text(json.dumps(status, indent=2))
    del model, optimizer, synth
    if device.type == "cuda": torch.cuda.empty_cache()
    return status

runs = pd.DataFrame([train_condition(name, duties, seed)
                     for name, duties in CONDITIONS.items() for seed in TRAIN_SEEDS])
runs.to_csv(OUT_DIR / "training_runs.csv", index=False)
display(runs)

## 5. Evaluate all models on the same duty sweep
Each sweep recording is generated once and scored by all six models. Scene seeds are disjoint from the training scenes.

In [ ]:
models = {}
for name in CONDITIONS:
    for seed in TRAIN_SEEDS:
        m = SegmentationModel(BASE_CHANNELS).to(device).eval()
        m.load_state_dict(torch.load(CHECKPOINT_DIR / f"{name}_seed{seed}" / "best.pt", map_location=device, weights_only=True))
        models[(name, seed)] = m

hists = {}
started = time.perf_counter()
with torch.inference_mode():
    for family in EVAL_FAMILIES:
        for scene in EVAL_SEEDS:
            for strength in EVAL_STRENGTHS:
                for duty in EVAL_DUTIES:
                    rec = simulate_recording(seed=scene, rfi=family, strength=strength, duty_cycle=duty)
                    x = (rec["power"][..., 0].transpose(1, 0, 2).astype(np.float32) - NORM["mean"]) / NORM["std"]
                    x = torch.from_numpy(x)[:, None].to(device)
                    truth = rec["rfi_mask"][..., 0].transpose(1, 0, 2)
                    for key, m in models.items():
                        q = np.floor(m(x).sigmoid()[:, 0].cpu().numpy() * (SWEEP_BINS - 1)).astype(np.int64)
                        h = hists.setdefault((*key, family, strength, duty), np.zeros((2, SWEEP_BINS), np.int64))
                        h[0] += np.bincount(q[truth], minlength=SWEEP_BINS)
                        h[1] += np.bincount(q[~truth], minlength=SWEEP_BINS)
        print(f"{family}: done, {time.perf_counter() - started:.0f} s", flush=True)

sweep = pd.DataFrame([dict(condition=c, train_seed=s, family=f, strength=st, duty_cycle=d, pixel_ap=histogram_ap(*h))
                      for (c, s, f, st, d), h in hists.items()])
sweep.to_csv(OUT_DIR / "sweep_ap.csv", index=False)
with pd.option_context("display.max_rows", 60, "display.float_format", "{:.4f}".format):
    display(sweep.pivot_table(index=["family", "strength", "duty_cycle"], columns="condition", values="pixel_ap"))

## 6. Plot: AP versus duty cycle, one line per training condition

In [ ]:
COLORS = {"low": "#2a78d6", "high": "#eb6834", "all": "#1baf7a"}
fig, axes = plt.subplots(len(EVAL_FAMILIES), len(EVAL_STRENGTHS), figsize=(13, 3.3 * len(EVAL_FAMILIES)),
                         sharex=True, layout="constrained")
for i, family in enumerate(EVAL_FAMILIES):
    for j, strength in enumerate(EVAL_STRENGTHS):
        ax = axes[i, j]
        for name, color in COLORS.items():
            d = sweep.query("family == @family and strength == @strength and condition == @name")
            d = d.groupby("duty_cycle").pixel_ap.agg(["mean", "min", "max"]).reset_index()
            ax.plot(d.duty_cycle, d["mean"], marker="o", ms=6, lw=2, color=color,
                    label=f"{name} {CONDITIONS[name]}")
            ax.fill_between(d.duty_cycle, d["min"], d["max"], color=color, alpha=.15, lw=0)
        ax.set_title(f"{family} · peak strength {strength:g}")
        ax.grid(alpha=.2)
    axes[i, 0].set_ylabel("Pixel AP (band: seed range)")
for ax in axes[-1]: ax.set_xlabel("Evaluation duty cycle")
axes[0, -1].legend(frameon=False, fontsize=9)
fig.savefig(OUT_DIR / "ap_vs_duty_by_training_mix.png", dpi=200)
plt.show(); plt.close(fig)

## 7. How to read the results
- Where the **low** line falls below **all** at high duty (and **high** below **all** at low duty), the model did not transfer across duty cycles; equal lines mean the duty mix does not matter.
- Duty 0.5 and 0.9 are unseen by both low and high; they test interpolation.
- If the band between the two training seeds overlaps another condition's line, that difference is not resolved; add seeds before claiming it.
- `training_runs.csv` also shows whether the duty mix changes performance on the ordinary starter validation set.
- Files in `OUT_DIR`: one folder per run (`history.csv`, `synthetic_train.csv`, `done.json`), plus `training_runs.csv`, `sweep_ap.csv` and the PNG. Each run's `best.pt` is in the same-named folder under `CHECKPOINT_DIR`.